# CHATBOT

In [ ]:
# =============================================================================
# RETAIL AI INTELLIGENCE PLATFORM — MODULE 1: AI CHAT ASSISTANT
# Paste this entire file into a Google Colab cell and run
# =============================================================================

# ─── CELL 1: INSTALL DEPENDENCIES ────────────────────────────────────────────
# Run this cell first, then restart runtime when prompted

import subprocess
subprocess.run(["pip", "install", "groq", "snowflake-connector-python",
                "streamlit", "plotly", "pandas", "numpy", "-q"])

print("✅ All dependencies installed")


✅ All dependencies installed


In [ ]:
# ─── FILL IN YOUR CREDENTIALS HERE ──────────────────────────
# Get your free Groq API key at: https://console.groq.com
GROQ_API_KEY = "your_groq_api_key_here"

SNOWFLAKE_CONFIG = {
    "account":   "your_snowflake_account_identifier",
    "user":      "your_snowflake_username",
    "password":  "your_snowflake_password",
    "warehouse": "COMPUTE_WH",
    "database":  "RETAIL_RAW",
    "schema":    "RAW_MARTS",
}

✅ Configuration set


In [ ]:

# ─── CELL 3: SNOWFLAKE CONNECTION ────────────────────────────────────────────

import snowflake.connector
import pandas as pd

def get_snowflake_connection():
    """Create and return a Snowflake connection."""
    conn = snowflake.connector.connect(
        account   = SNOWFLAKE_CONFIG["account"],
        user      = SNOWFLAKE_CONFIG["user"],
        password  = SNOWFLAKE_CONFIG["password"],
        warehouse = SNOWFLAKE_CONFIG["warehouse"],
        database  = SNOWFLAKE_CONFIG["database"],
        schema    = SNOWFLAKE_CONFIG["schema"],
    )
    return conn

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query and return results as a DataFrame."""
    conn = get_snowflake_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(sql)
        cols = [desc[0].lower() for desc in cursor.description]
        rows = cursor.fetchall()
        return pd.DataFrame(rows, columns=cols)
    finally:
        conn.close()

# Test connection
try:
    test = run_query("SELECT COUNT(*) as total_rows FROM MART_SALES_FACT")
    print(f"✅ Snowflake connected — {test['total_rows'][0]:,} rows in MART_SALES_FACT")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Snowflake connected — 60,699 rows in MART_SALES_FACT


In [ ]:

# ─── CELL 4: SCHEMA CONTEXT FOR THE LLM ─────────────────────────────────────
# This tells the LLM exactly what tables and columns exist
# so it writes correct SQL every time

SCHEMA_CONTEXT = """
You are a senior retail data analyst with access to a Snowflake data warehouse.
You help business users answer questions about retail sales data.

You have access to these tables in schema RAW_MARTS:

TABLE: MART_SALES_FACT
One row per order line item. Core fact table.
Columns:
- sale_key          : unique identifier per line item
- order_number      : order ID (multiple line items per order)
- line_item         : line number within order
- order_date        : date of order (DATE type)
- delivery_date     : date delivered (NULL for online orders)
- customer_key      : FK to MART_DIM_CUSTOMER
- store_key         : FK to MART_DIM_STORE (0 = Online)
- product_key       : FK to MART_DIM_PRODUCT
- channel           : 'Online' or 'In-Store'
- customer_name     : customer full name
- gender            : Male/Female
- customer_country  : customer's country
- continent         : customer's continent
- age_band          : Under 35 / 35-49 / 50-64 / 65+
- store_country     : store's country
- store_state       : store's state
- store_type        : Online or Physical
- store_size_band   : Small/Medium/Large/XLarge
- product_name      : full product name
- brand             : product brand
- category          : product category (Computers, Audio, etc.)
- subcategory       : product subcategory
- currency_code     : USD/CAD/AUD/EUR/GBP
- exchange_rate     : FX rate used
- quantity          : units purchased
- unit_price_usd    : list price in USD
- unit_cost_usd     : cost in USD
- revenue_usd       : actual USD revenue (quantity * price / fx_rate)
- cost_usd          : actual USD cost
- profit_usd        : revenue_usd - cost_usd
- margin_pct        : profit as % of revenue
- delivery_days     : days from order to delivery
- order_year        : year of order
- order_month_num   : month number (1-12)

TABLE: MART_DIM_CUSTOMER
One row per customer.
Columns:
- customer_key, customer_name, gender, city, state
- country, continent, birthday, customer_age_years, age_band
- total_revenue_usd  : customer lifetime value
- total_orders       : number of orders placed
- total_units        : total units bought
- avg_order_value_usd: average order value
- last_order_date    : most recent order date
- days_since_last_order: recency in days
- rfm_segment        : Champions/Loyal/At Risk/Hibernating/Lost etc
- r_score, f_score, m_score: RFM scores 1-5
- has_orders         : TRUE/FALSE

TABLE: MART_DIM_PRODUCT
One row per product.
Columns:
- product_key, product_name, brand, color
- category, subcategory
- unit_cost_usd, unit_price_usd, list_margin_pct
- total_revenue_usd, total_units_sold, total_orders
- abc_class: A — Top (70%) / B — Mid (20%) / C — Tail (10%)

TABLE: MART_DIM_STORE
One row per store (StoreKey=0 is Online channel).
Columns:
- store_key, country, state, store_type, square_meters
- size_band, store_age_years, open_date
- total_revenue_usd, total_orders, unique_customers
- revenue_per_sqm, annualized_revenue_usd, revenue_rank

TABLE: MART_EXECUTIVE_SUMMARY
Pre-aggregated monthly KPIs.
Columns:
- year, month_num, month_start, store_country
- category, channel
- revenue_usd, profit_usd, margin_pct
- total_orders, total_units, unique_customers
- avg_order_value_usd
- yoy_growth_pct: year over year growth %
- mom_growth_pct: month over month growth %

IMPORTANT RULES:
1. Always write Snowflake-compatible SQL
2. Always use schema prefix RAW_MARTS (e.g. RAW_MARTS.MART_SALES_FACT)
3. Always LIMIT results to max 20 rows unless asked for more
4. For revenue always use revenue_usd column
5. For dates use order_date column and YEAR(), MONTH() functions
6. Always round monetary values to 2 decimal places
7. Return ONLY the SQL query — no explanation, no markdown, no backticks
8. If question cannot be answered with available data, return: CANNOT_ANSWER
"""

print("✅ Schema context loaded")


✅ Schema context loaded


In [ ]:
# ─── CELL 5: GROQ LLM FUNCTIONS ──────────────────────────────────────────────

from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def generate_sql(user_question: str) -> str:
    """Send user question to Groq LLM → get SQL query back."""

    prompt = f"""
{SCHEMA_CONTEXT}

User question: {user_question}

Write a SQL query to answer this question.
Return ONLY the SQL — nothing else.
"""

    response = groq_client.chat.completions.create(
        model       = "llama-3.3-70b-versatile",
        messages    = [{"role": "user", "content": prompt}],
        temperature = 0.1,
        max_tokens  = 500,
    )

    sql = response.choices[0].message.content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


def generate_answer(user_question: str, sql: str, df) -> str:
    """Send query results to Groq LLM → get human readable answer."""

    if df.empty:
        data_str = "No results found."
    else:
        data_str = df.to_string(index=False, max_rows=20)

    prompt = f"""
You are a helpful retail business analyst.

The user asked: "{user_question}"

The SQL query returned this data:
{data_str}

Write a clear, concise business answer in 2-4 sentences.
Include actual numbers. Be specific and insightful.
Do not mention SQL or technical details.
"""

    response = groq_client.chat.completions.create(
        model       = "llama-3.3-70b-versatile",
        messages    = [{"role": "user", "content": prompt}],
        temperature = 0.3,
        max_tokens  = 300,
    )

    return response.choices[0].message.content.strip()


# Test the LLM connection
try:
    test_sql = generate_sql("How many total orders are there?")
    print(f"✅ Groq connected — test SQL generated:\n{test_sql}")
except Exception as e:
    print(f"❌ Groq connection failed: {e}")

✅ Groq connected — test SQL generated:
SELECT COUNT(DISTINCT order_number) FROM RAW_MARTS.MART_SALES_FACT LIMIT 20


In [ ]:

# ─── CELL 6: MAIN CHAT FUNCTION ──────────────────────────────────────────────

def ask_retail_ai(question: str, verbose: bool = True) -> dict:
    """
    Main function — takes a plain English question,
    returns SQL + dataframe + human readable answer.

    Args:
        question: plain English business question
        verbose:  if True, prints each step

    Returns:
        dict with keys: question, sql, data, answer, success
    """

    result = {
        "question": question,
        "sql":      None,
        "data":     None,
        "answer":   None,
        "success":  False,
    }

    try:
        # Step 1: Generate SQL
        if verbose:
            print(f"\n{'='*60}")
            print(f"❓ Question: {question}")
            print(f"{'='*60}")
            print("🔄 Generating SQL...")

        sql = generate_sql(question)
        result["sql"] = sql

        if sql == "CANNOT_ANSWER":
            result["answer"] = "I cannot answer this question with the available data."
            return result

        if verbose:
            print(f"📝 Generated SQL:\n{sql}\n")

        # Step 2: Run SQL against Snowflake
        if verbose:
            print("🔄 Querying Snowflake...")

        df = run_query(sql)
        result["data"] = df

        if verbose:
            print(f"📊 Results ({len(df)} rows):")
            print(df.to_string(index=False))
            print()

        # Step 3: Generate human readable answer
        if verbose:
            print("🔄 Generating answer...")

        answer = generate_answer(question, sql, df)
        result["answer"]  = answer
        result["success"] = True

        if verbose:
            print(f"💡 Answer:\n{answer}")

    except Exception as e:
        result["answer"] = f"Error: {str(e)}"
        if verbose:
            print(f"❌ Error: {e}")

    return result


print("✅ Chat function ready")


✅ Chat function ready


In [ ]:

# ─── CELL 7: TEST WITH SAMPLE QUESTIONS ──────────────────────────────────────

print("\n" + "="*60)
print("TESTING RETAIL AI CHAT ASSISTANT")
print("="*60)

test_questions = [
    "What is the total revenue across all years?",
    "Which product category generated the most revenue?",
    "Which country had the highest number of orders?",
    "What are the top 5 products by revenue?",
    "How many customers are in the Champions RFM segment?",
    "What was the revenue in 2019 vs 2020?",
    "Which store had the highest revenue per square meter?",
]

# Run first 3 questions as test
for q in test_questions[:3]:
    result = ask_retail_ai(q, verbose=True)
    print()




TESTING RETAIL AI CHAT ASSISTANT

❓ Question: What is the total revenue across all years?
🔄 Generating SQL...
📝 Generated SQL:
SELECT ROUND(SUM(revenue_usd), 2) AS total_revenue FROM RAW_MARTS.MART_SALES_FACT LIMIT 1

🔄 Querying Snowflake...
📊 Results (1 rows):
 total_revenue
   44611749.69

🔄 Generating answer...
💡 Answer:
The total revenue across all years is approximately $44.6 million. This figure represents the cumulative revenue generated by our business over the entire period. With total revenues reaching $44,611,749.69, we can see that our business has achieved significant sales. This insight provides a solid foundation for further analysis and strategic decision-making to drive future growth.


❓ Question: Which product category generated the most revenue?
🔄 Generating SQL...
📝 Generated SQL:
SELECT category, ROUND(SUM(revenue_usd), 2) AS total_revenue FROM RAW_MARTS.MART_SALES_FACT GROUP BY category ORDER BY total_revenue DESC LIMIT 1

🔄 Querying Snowflake...
📊 Results (1 ro

In [ ]:
# ─── CELL 8: INTERACTIVE CHAT LOOP ───────────────────────────────────────────
# Run this cell for an interactive session in Colab

print("\n" + "="*60)
print("💬 RETAIL AI CHAT — Type your questions below")
print("   Type 'quit' to exit")
print("="*60 + "\n")

while True:
    try:
        question = input("You: ").strip()

        if not question:
            continue

        if question.lower() in ["quit", "exit", "q"]:
            print("👋 Goodbye!")
            break

        result = ask_retail_ai(question, verbose=False)

        print(f"\n💡 Answer: {result['answer']}")

        # Ask if user wants to see the data
        show_data = input("\nShow raw data? (y/n): ").strip().lower()
        if show_data == "y" and result["data"] is not None:
            print("\n📊 Data:")
            print(result["data"].to_string(index=False))

        # Ask if user wants to see the SQL
        show_sql = input("Show SQL? (y/n): ").strip().lower()
        if show_sql == "y" and result["sql"]:
            print(f"\n📝 SQL:\n{result['sql']}")

        print("\n" + "-"*60 + "\n")

    except KeyboardInterrupt:
        print("\n👋 Chat ended.")
        break


💬 RETAIL AI CHAT — Type your questions below
   Type 'quit' to exit

You: Which month historically has the highest revenue?

💡 Answer: Historically, our highest revenue month is February, with total revenues reaching $6,102,464.86. This suggests that our business experiences a significant surge in sales during this period, potentially due to post-holiday shopping or winter promotions. By understanding this trend, we can optimize our inventory and marketing strategies to capitalize on this opportunity and drive even greater sales growth in February.

Show raw data? (y/n): Y

📊 Data:
 month_num  total_revenue
         2     6102464.86
Show SQL? (y/n): Y

📝 SQL:
SELECT MONTH(order_date) AS month_num, ROUND(SUM(revenue_usd), 2) AS total_revenue 
FROM RAW_MARTS.MART_SALES_FACT 
GROUP BY MONTH(order_date) 
ORDER BY total_revenue DESC 
LIMIT 1

------------------------------------------------------------

You: Which store has the highest revenue per square meter?

💡 Answer: Error: 000904 (42

# FORECASTING

In [ ]:
# =============================================================================
# RETAIL AI INTELLIGENCE PLATFORM — MODULE 2: SALES FORECASTING ENGINE
# Paste this into a NEW cell block AFTER Module 1 cells in your Colab
# Requires: Module 1 cells already run (Snowflake connection + config)
# =============================================================================

# ─── CELL 9: INSTALL FORECASTING DEPENDENCIES ────────────────────────────────

import subprocess
subprocess.run(["pip", "install", "prophet", "plotly", "-q"])
print("✅ Prophet and Plotly installed")



✅ Prophet and Plotly installed


In [ ]:
# ─── CELL 10: FORECASTING DATA LOADER ────────────────────────────────────────

import pandas as pd
import numpy as np
from datetime import datetime

def load_forecast_data(
    forecast_by: str = "total",
    filter_value: str = None
) -> pd.DataFrame:
    """
    Load monthly revenue data from Snowflake for forecasting.

    Args:
        forecast_by:  'total' | 'category' | 'country' | 'channel'
        filter_value: specific value to filter on (e.g. 'Computers')

    Returns:
        DataFrame with columns: ds (date), y (revenue)
        Prophet requires exactly these column names
    """

    if forecast_by == "total":
        sql = """
            SELECT
                DATE_TRUNC('month', order_date)  AS ds,
                ROUND(SUM(revenue_usd), 2)        AS y
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE YEAR(order_date) < 2021
            GROUP BY DATE_TRUNC('month', order_date)
            ORDER BY ds
        """

    elif forecast_by == "category":
        sql = f"""
            SELECT
                DATE_TRUNC('month', order_date)  AS ds,
                ROUND(SUM(revenue_usd), 2)        AS y
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE YEAR(order_date) < 2021
              AND category = '{filter_value}'
            GROUP BY DATE_TRUNC('month', order_date)
            ORDER BY ds
        """

    elif forecast_by == "country":
        sql = f"""
            SELECT
                DATE_TRUNC('month', order_date)  AS ds,
                ROUND(SUM(revenue_usd), 2)        AS y
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE YEAR(order_date) < 2021
              AND store_country = '{filter_value}'
            GROUP BY DATE_TRUNC('month', order_date)
            ORDER BY ds
        """

    elif forecast_by == "channel":
        sql = f"""
            SELECT
                DATE_TRUNC('month', order_date)  AS ds,
                ROUND(SUM(revenue_usd), 2)        AS y
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE YEAR(order_date) < 2021
              AND channel = '{filter_value}'
            GROUP BY DATE_TRUNC('month', order_date)
            ORDER BY ds
        """

    df = run_query(sql)
    df["ds"] = pd.to_datetime(df["ds"])
    df["y"]  = pd.to_numeric(df["y"], errors="coerce")
    df       = df.dropna()

    return df


def get_filter_options(forecast_by: str) -> list:
    """Get available filter values for a given forecast dimension."""

    if forecast_by == "category":
        sql = """
            SELECT DISTINCT category
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE category IS NOT NULL
            ORDER BY category
        """
    elif forecast_by == "country":
        sql = """
            SELECT DISTINCT store_country
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE store_country IS NOT NULL
            ORDER BY store_country
        """
    elif forecast_by == "channel":
        sql = """
            SELECT DISTINCT channel
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE channel IS NOT NULL
            ORDER BY channel
        """
    else:
        return ["total"]

    df = run_query(sql)
    return df.iloc[:, 0].tolist()


# Test data loader
print("Testing data loader...")
test_df = load_forecast_data("total")
print(f"✅ Loaded {len(test_df)} monthly data points")
print(f"   Date range: {test_df['ds'].min().date()} → {test_df['ds'].max().date()}")
print(f"   Revenue range: ${test_df['y'].min():,.0f} → ${test_df['y'].max():,.0f}")


Testing data loader...
✅ Loaded 60 monthly data points
   Date range: 2016-01-01 → 2020-12-01
   Revenue range: $30,832 → $2,045,390


In [ ]:

# ─── CELL 11: PROPHET FORECASTING ENGINE ─────────────────────────────────────

from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")

def run_forecast(
    forecast_by:   str = "total",
    filter_value:  str = None,
    months_ahead:  int = 6,
    include_covid: bool = False
) -> dict:
    """
    Run Prophet forecast on retail revenue data.

    Args:
        forecast_by:   'total' | 'category' | 'country' | 'channel'
        filter_value:  specific value to filter on
        months_ahead:  how many months to forecast
        include_covid: if False, adds COVID lockdown as a holiday/event

    Returns:
        dict with: df_history, df_forecast, model, metrics, label
    """

    # Build label for display
    if forecast_by == "total":
        label = "All Revenue"
    else:
        label = f"{forecast_by.title()}: {filter_value}"

    print(f"\n{'='*55}")
    print(f"📈 FORECASTING: {label}")
    print(f"   Horizon: {months_ahead} months ahead")
    print(f"{'='*55}")

    # Step 1: Load data
    print("🔄 Loading historical data from Snowflake...")
    df = load_forecast_data(forecast_by, filter_value)
    print(f"✅ {len(df)} months of history loaded")

    # Step 2: Configure Prophet model
    model = Prophet(
        yearly_seasonality  = True,
        weekly_seasonality  = False,   # monthly data — no weekly pattern
        daily_seasonality   = False,
        seasonality_mode    = "multiplicative",
        changepoint_prior_scale = 0.05,  # flexibility of trend
        interval_width      = 0.95,      # 95% confidence intervals
    )

    # Add COVID lockdown as a known shock event
    # This prevents the model from learning the 2020 drop as "normal"
    if not include_covid:
        covid_lockdown = pd.DataFrame({
            "holiday":    "covid_lockdown",
            "ds":         pd.to_datetime([
                "2020-03-01", "2020-04-01", "2020-05-01",
                "2020-06-01", "2020-07-01", "2020-08-01"
            ]),
            "lower_window": 0,
            "upper_window": 1,
        })
        model = Prophet(
            yearly_seasonality      = True,
            weekly_seasonality      = False,
            daily_seasonality       = False,
            seasonality_mode        = "multiplicative",
            changepoint_prior_scale = 0.05,
            interval_width          = 0.95,
            holidays                = covid_lockdown,
        )

    # Step 3: Fit model
    print("🔄 Training Prophet model...")
    model.fit(df)
    print("✅ Model trained")

    # Step 4: Create future dataframe
    future = model.make_future_dataframe(
        periods = months_ahead,
        freq    = "MS"  # Month Start frequency
    )

    # Step 5: Generate forecast
    print(f"🔄 Forecasting {months_ahead} months ahead...")
    forecast = model.predict(future)

    # Step 6: Calculate accuracy metrics on historical data
    df_merged = df.merge(
        forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]],
        on = "ds"
    )

    # MAPE — Mean Absolute Percentage Error
    mape = (
        abs(df_merged["y"] - df_merged["yhat"]) / df_merged["y"]
    ).mean() * 100

    # MAE — Mean Absolute Error
    mae = abs(df_merged["y"] - df_merged["yhat"]).mean()

    print(f"\n📊 MODEL ACCURACY:")
    print(f"   MAPE : {mape:.1f}% (lower is better)")
    print(f"   MAE  : ${mae:,.0f} per month")

    # Step 7: Extract future predictions only
    df_future = forecast[forecast["ds"] > df["ds"].max()].copy()
    df_future = df_future[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
    df_future.columns = ["date", "predicted_revenue", "lower_bound", "upper_bound"]
    df_future["predicted_revenue"] = df_future["predicted_revenue"].round(2)
    df_future["lower_bound"]       = df_future["lower_bound"].round(2)
    df_future["upper_bound"]       = df_future["upper_bound"].round(2)

    print(f"\n📅 FORECAST RESULTS ({months_ahead} months):")
    print(df_future[["date", "predicted_revenue", "lower_bound", "upper_bound"]]
          .to_string(index=False))

    return {
        "label":       label,
        "df_history":  df,
        "df_forecast": df_future,
        "full_forecast": forecast,
        "model":       model,
        "mape":        mape,
        "mae":         mae,
    }


# Test forecast
result = run_forecast(
    forecast_by  = "total",
    months_ahead = 6
)
print("\n✅ Forecasting engine working!")


📈 FORECASTING: All Revenue
   Horizon: 6 months ahead
🔄 Loading historical data from Snowflake...
✅ 60 months of history loaded
🔄 Training Prophet model...
✅ Model trained
🔄 Forecasting 6 months ahead...

📊 MODEL ACCURACY:
   MAPE : 57.4% (lower is better)
   MAE  : $221,139 per month

📅 FORECAST RESULTS (6 months):
      date  predicted_revenue  lower_bound  upper_bound
2021-01-01         2377517.33   1792002.01   2941469.10
2021-02-01         2080794.48   1495658.44   2623042.53
2021-03-01          989244.71    414670.71   1583554.81
2021-04-01          445317.48   -160968.56   1020077.44
2021-05-01         1502494.41    912303.81   2151638.50
2021-06-01         1438677.98    832778.31   1972675.85

✅ Forecasting engine working!


In [ ]:
# ─── CELL 12: PLOTLY VISUALISATION (FIXED) ───────────────────────────────────

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_forecast(forecast_result: dict, show_components: bool = True):
    """
    Create an interactive Plotly chart for the forecast.
    Shows: historical data + forecast + confidence interval
    """

    df_hist   = forecast_result["df_history"]
    df_future = forecast_result["df_forecast"]
    full_fc   = forecast_result["full_forecast"]
    label     = forecast_result["label"]
    mape      = forecast_result["mape"]

    # Combine historical fitted values
    df_fitted = full_fc[full_fc["ds"] <= df_hist["ds"].max()].copy()

    # Convert all dates to strings — avoids Plotly Timestamp bug
    hist_dates   = df_hist["ds"].dt.strftime("%Y-%m-%d").tolist()
    fitted_dates = df_fitted["ds"].dt.strftime("%Y-%m-%d").tolist()
    future_dates = df_future["date"].dt.strftime("%Y-%m-%d").tolist()
    forecast_start_str = future_dates[0]

    if show_components:
        fig = make_subplots(
            rows = 2, cols = 1,
            subplot_titles = (
                f"Revenue Forecast — {label}",
                "Yearly Seasonality Pattern"
            ),
            row_heights = [0.7, 0.3],
            vertical_spacing = 0.12
        )
    else:
        fig = go.Figure()

    row = 1 if show_components else None
    rc  = {"row": row, "col": 1} if show_components else {}

    # ── Historical confidence interval band ───────────────────────────────────
    fig.add_trace(go.Scatter(
        x         = fitted_dates + fitted_dates[::-1],
        y         = df_fitted["yhat_upper"].tolist() + df_fitted["yhat_lower"].tolist()[::-1],
        fill      = "toself",
        fillcolor = "rgba(37, 99, 235, 0.1)",
        line      = dict(color="rgba(255,255,255,0)"),
        name      = "Historical CI",
        showlegend= False,
    ), **rc)

    # ── Future confidence interval band ──────────────────────────────────────
    fig.add_trace(go.Scatter(
        x         = future_dates + future_dates[::-1],
        y         = df_future["upper_bound"].tolist() + df_future["lower_bound"].tolist()[::-1],
        fill      = "toself",
        fillcolor = "rgba(245, 158, 11, 0.2)",
        line      = dict(color="rgba(255,255,255,0)"),
        name      = "Forecast CI (95%)",
        showlegend= True,
    ), **rc)

    # ── Actual historical revenue ─────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x    = hist_dates,
        y    = df_hist["y"].tolist(),
        mode = "lines+markers",
        name = "Actual Revenue",
        line = dict(color="#2563EB", width=2),
        marker = dict(size=5),
    ), **rc)

    # ── Prophet fitted line ───────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x    = fitted_dates,
        y    = df_fitted["yhat"].tolist(),
        mode = "lines",
        name = "Model Fit",
        line = dict(color="#7C3AED", width=1.5, dash="dot"),
    ), **rc)

    # ── Forecast line ─────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x    = future_dates,
        y    = df_future["predicted_revenue"].tolist(),
        mode = "lines+markers",
        name = "Forecast",
        line = dict(color="#F59E0B", width=3),
        marker = dict(size=8, symbol="diamond"),
    ), **rc)

    # ── Vertical line at forecast start (using shape instead of add_vline) ────
    fig.add_shape(
        type  = "line",
        xref  = "x",
        yref  = "paper",
        x0    = forecast_start_str,
        x1    = forecast_start_str,
        y0    = 0,
        y1    = 1,
        line  = dict(color="#EF4444", dash="dash", width=2),
        row   = row,
        col   = 1,
    )

    # Add annotation for forecast start separately
    fig.add_annotation(
        x    = forecast_start_str,
        y    = 1,
        xref = "x",
        yref = "paper",
        text = "Forecast Start",
        showarrow = False,
        font = dict(color="#EF4444", size=11),
        xanchor = "left",
        row = row,
        col = 1,
    )

    # ── Seasonality component ─────────────────────────────────────────────────
    if show_components:
        yearly_input = pd.DataFrame({
            "ds": pd.date_range("2019-01-01", "2019-12-31", freq="MS")
        })
        yearly = forecast_result["model"].predict(yearly_input)
        yearly_dates = yearly["ds"].dt.strftime("%Y-%m-%d").tolist()

        fig.add_trace(go.Scatter(
            x         = yearly_dates,
            y         = yearly["yearly"].tolist(),
            mode      = "lines+markers",
            name      = "Yearly Seasonality",
            line      = dict(color="#10B981", width=2),
            fill      = "tozeroy",
            fillcolor = "rgba(16, 185, 129, 0.1)",
        ), row=2, col=1)

        fig.update_xaxes(title_text="Month",           row=2, col=1)
        fig.update_yaxes(title_text="Seasonal Effect", row=2, col=1)

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        title = dict(
            text = (
                f"📈 Sales Forecast — {label}<br>"
                f"<sup>MAPE: {mape:.1f}% | 95% Confidence Intervals shown</sup>"
            ),
            font = dict(size=16),
        ),
        height        = 700 if show_components else 500,
        hovermode     = "x unified",
        legend        = dict(orientation="h", yanchor="bottom", y=1.02),
        plot_bgcolor  = "white",
        paper_bgcolor = "white",
    )

    fig.update_yaxes(
        title_text = "Revenue (USD)",
        tickformat = "$,.0f",
        row=1, col=1,
    )
    fig.update_xaxes(
        title_text = "Date",
        row=1, col=1,
    )

    fig.show()
    return fig


# Plot the test forecast
fig = plot_forecast(result)
print("✅ Chart displayed")

✅ Chart displayed


In [ ]:

# ─── CELL 13: FORECAST WITH SELECTOR ─────────────────────────────────────────
# Interactive selector — choose what to forecast

print("\n" + "="*55)
print("📈 SALES FORECASTING ENGINE — Interactive Mode")
print("="*55)

# Show available options
print("\nForecast dimensions:")
print("  1 → Total Revenue")
print("  2 → By Category")
print("  3 → By Country")
print("  4 → By Channel")

choice = input("\nSelect dimension (1-4): ").strip()

forecast_by_map = {
    "1": "total",
    "2": "category",
    "3": "country",
    "4": "channel",
}

forecast_by = forecast_by_map.get(choice, "total")

# Show filter options if needed
filter_value = None
if forecast_by != "total":
    options = get_filter_options(forecast_by)
    print(f"\nAvailable {forecast_by} options:")
    for i, opt in enumerate(options, 1):
        print(f"  {i} → {opt}")

    opt_choice = input(f"\nSelect {forecast_by} (enter number): ").strip()
    try:
        filter_value = options[int(opt_choice) - 1]
    except Exception:
        filter_value = options[0]

# Select months ahead
months_input = input("\nHow many months to forecast? (default 6): ").strip()
months_ahead = int(months_input) if months_input.isdigit() else 6

# Run forecast
forecast_result = run_forecast(
    forecast_by  = forecast_by,
    filter_value = filter_value,
    months_ahead = months_ahead,
)

# Plot results
fig = plot_forecast(forecast_result)

# Summary table
print(f"\n📋 FORECAST SUMMARY — {forecast_result['label']}")
print("="*55)
summary = forecast_result["df_forecast"].copy()
summary["date"]              = summary["date"].dt.strftime("%b %Y")
summary["predicted_revenue"] = summary["predicted_revenue"].apply(lambda x: f"${x:,.0f}")
summary["lower_bound"]       = summary["lower_bound"].apply(lambda x: f"${x:,.0f}")
summary["upper_bound"]       = summary["upper_bound"].apply(lambda x: f"${x:,.0f}")
summary.columns              = ["Month", "Predicted Revenue", "Low Estimate", "High Estimate"]
print(summary.to_string(index=False))
print(f"\nModel Accuracy (MAPE): {forecast_result['mape']:.1f}%")


📈 SALES FORECASTING ENGINE — Interactive Mode

Forecast dimensions:
  1 → Total Revenue
  2 → By Category
  3 → By Country
  4 → By Channel

Select dimension (1-4): 4

Available channel options:
  1 → In-Store
  2 → Online

Select channel (enter number): 1

How many months to forecast? (default 6): 6

📈 FORECASTING: Channel: In-Store
   Horizon: 6 months ahead
🔄 Loading historical data from Snowflake...
✅ 60 months of history loaded
🔄 Training Prophet model...
✅ Model trained
🔄 Forecasting 6 months ahead...

📊 MODEL ACCURACY:
   MAPE : 56.5% (lower is better)
   MAE  : $173,123 per month

📅 FORECAST RESULTS (6 months):
      date  predicted_revenue  lower_bound  upper_bound
2021-01-01         1864433.58   1400731.77   2290530.60
2021-02-01         1640175.61   1170580.09   2107199.92
2021-03-01          761918.77    301580.87   1249733.47
2021-04-01          340492.96   -141896.03    803050.41
2021-05-01         1172426.26    743514.85   1607275.65
2021-06-01         1133634.75    678


📋 FORECAST SUMMARY — Channel: In-Store
   Month Predicted Revenue Low Estimate High Estimate
Jan 2021        $1,864,434   $1,400,732    $2,290,531
Feb 2021        $1,640,176   $1,170,580    $2,107,200
Mar 2021          $761,919     $301,581    $1,249,733
Apr 2021          $340,493    $-141,896      $803,050
May 2021        $1,172,426     $743,515    $1,607,276
Jun 2021        $1,133,635     $678,767    $1,598,966

Model Accuracy (MAPE): 56.5%


# ANOMALY DETECTION

In [ ]:
# =============================================================================
# RETAIL AI INTELLIGENCE PLATFORM — MODULE 3: ANOMALY DETECTION ENGINE
# Paste these as NEW cells AFTER Module 1 and Module 2 cells in Colab
# Requires: Module 1 cells already run (Snowflake connection + config)
# =============================================================================

# ─── CELL 14: INSTALL ANOMALY DETECTION DEPENDENCIES ─────────────────────────

import subprocess
subprocess.run(["pip", "install", "scikit-learn", "plotly", "-q"])
print("✅ Scikit-learn and Plotly ready")


✅ Scikit-learn and Plotly ready


In [ ]:

# ─── CELL 15: LOAD DATA FOR ANOMALY DETECTION ────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

def load_anomaly_data(level: str = "monthly") -> pd.DataFrame:
    """
    Load aggregated sales data for anomaly detection.

    Args:
        level: 'monthly' | 'store' | 'product' | 'category'

    Returns:
        DataFrame ready for anomaly detection
    """

    if level == "monthly":
        sql = """
            SELECT
                DATE_TRUNC('month', order_date)     AS period,
                'Overall'                            AS dimension,
                ROUND(SUM(revenue_usd), 2)           AS revenue,
                COUNT(DISTINCT order_number)         AS order_count,
                ROUND(AVG(revenue_usd), 2)           AS avg_line_revenue,
                SUM(quantity)                        AS total_units,
                ROUND(AVG(margin_pct), 2)            AS avg_margin,
                COUNT(DISTINCT customer_key)         AS unique_customers
            FROM RAW_MARTS.MART_SALES_FACT
            GROUP BY DATE_TRUNC('month', order_date)
            ORDER BY period
        """

    elif level == "store":
        sql = """
            SELECT
                DATE_TRUNC('month', order_date)     AS period,
                store_country || ' - ' || store_state AS dimension,
                ROUND(SUM(revenue_usd), 2)           AS revenue,
                COUNT(DISTINCT order_number)         AS order_count,
                ROUND(AVG(revenue_usd), 2)           AS avg_line_revenue,
                SUM(quantity)                        AS total_units,
                ROUND(AVG(margin_pct), 2)            AS avg_margin,
                COUNT(DISTINCT customer_key)         AS unique_customers
            FROM RAW_MARTS.MART_SALES_FACT
            WHERE store_type = 'Physical'
            GROUP BY 1, 2
            ORDER BY 1, 2
        """

    elif level == "category":
        sql = """
            SELECT
                DATE_TRUNC('month', order_date)     AS period,
                category                             AS dimension,
                ROUND(SUM(revenue_usd), 2)           AS revenue,
                COUNT(DISTINCT order_number)         AS order_count,
                ROUND(AVG(revenue_usd), 2)           AS avg_line_revenue,
                SUM(quantity)                        AS total_units,
                ROUND(AVG(margin_pct), 2)            AS avg_margin,
                COUNT(DISTINCT customer_key)         AS unique_customers
            FROM RAW_MARTS.MART_SALES_FACT
            GROUP BY 1, 2
            ORDER BY 1, 2
        """

    elif level == "product":
        sql = """
            SELECT
                DATE_TRUNC('month', order_date)     AS period,
                product_name                         AS dimension,
                ROUND(SUM(revenue_usd), 2)           AS revenue,
                COUNT(DISTINCT order_number)         AS order_count,
                ROUND(AVG(revenue_usd), 2)           AS avg_line_revenue,
                SUM(quantity)                        AS total_units,
                ROUND(AVG(margin_pct), 2)            AS avg_margin,
                COUNT(DISTINCT customer_key)         AS unique_customers
            FROM RAW_MARTS.MART_SALES_FACT
            GROUP BY 1, 2
            ORDER BY 1, 2
        """

    df = run_query(sql)
    df["period"] = pd.to_datetime(df["period"])

    # Cast numeric columns
    for col in ["revenue","order_count","avg_line_revenue",
                "total_units","avg_margin","unique_customers"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.dropna()


# Test data loader
print("Testing anomaly data loader...")
test_df = load_anomaly_data("monthly")
print(f"✅ Loaded {len(test_df)} monthly records")
print(f"   Columns: {list(test_df.columns)}")
print(test_df.head(3).to_string(index=False))


Testing anomaly data loader...
✅ Loaded 62 monthly records
   Columns: ['period', 'dimension', 'revenue', 'order_count', 'avg_line_revenue', 'total_units', 'avg_margin', 'unique_customers']
    period dimension   revenue  order_count  avg_line_revenue  total_units  avg_margin  unique_customers
2016-01-01   Overall 456907.36          284            722.95         2008       54.83               281
2016-02-01   Overall 585274.93          330            764.07         2488       54.48               325
2016-03-01   Overall 205590.25          106            799.96          818       54.59               106


In [ ]:

# ─── CELL 16: ISOLATION FOREST ANOMALY DETECTOR ──────────────────────────────

def detect_anomalies(
    df:               pd.DataFrame,
    features:         list = None,
    contamination:    float = 0.05,
    dimension_filter: str = None,
) -> pd.DataFrame:
    """
    Run Isolation Forest anomaly detection on sales data.

    Args:
        df:               input dataframe from load_anomaly_data()
        features:         columns to use for detection
        contamination:    expected % of anomalies (0.05 = 5%)
        dimension_filter: filter to specific store/category

    Returns:
        DataFrame with anomaly scores and flags added
    """

    # Filter to specific dimension if requested
    if dimension_filter and "dimension" in df.columns:
        df = df[df["dimension"] == dimension_filter].copy()

    if len(df) < 10:
        print(f"⚠️  Not enough data points ({len(df)}) for reliable detection.")
        return df

    # Default features if not specified
    if features is None:
        features = ["revenue", "order_count", "total_units", "avg_margin"]

    # Only use features that exist in dataframe
    features = [f for f in features if f in df.columns]

    # Scale features
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(df[features])

    # Train Isolation Forest
    model = IsolationForest(
        contamination = contamination,
        random_state  = 42,
        n_estimators  = 100,
    )

    df = df.copy()
    df["anomaly_score"] = model.fit_predict(X_scaled)
    df["anomaly_raw"]   = model.score_samples(X_scaled)

    # -1 = anomaly, 1 = normal → convert to readable flag
    df["is_anomaly"]    = df["anomaly_score"] == -1

    # Severity score (lower raw score = more anomalous)
    df["severity"] = pd.cut(
        df["anomaly_raw"],
        bins   = [-np.inf, -0.6, -0.5, -0.4, np.inf],
        labels = ["🔴 Critical", "🟠 High", "🟡 Medium", "🟢 Normal"]
    )

    # Revenue deviation from rolling mean
    df = df.sort_values("period")
    df["rolling_mean"]   = df["revenue"].rolling(window=3, min_periods=1).mean()
    df["revenue_deviation_pct"] = (
        (df["revenue"] - df["rolling_mean"]) / df["rolling_mean"] * 100
    ).round(1)

    return df


def summarize_anomalies(df: pd.DataFrame) -> pd.DataFrame:
    """Extract and summarize detected anomalies."""

    anomalies = df[df["is_anomaly"] == True].copy()

    if anomalies.empty:
        print("✅ No anomalies detected in this dataset.")
        return anomalies

    summary = anomalies[[
        "period", "dimension", "revenue", "order_count",
        "revenue_deviation_pct", "severity"
    ]].copy() if "dimension" in anomalies.columns else anomalies[[
        "period", "revenue", "order_count",
        "revenue_deviation_pct", "severity"
    ]].copy()

    summary = summary.sort_values("revenue_deviation_pct")
    summary["period"]   = summary["period"].dt.strftime("%b %Y")
    summary["revenue"]  = summary["revenue"].apply(lambda x: f"${x:,.0f}")
    summary["revenue_deviation_pct"] = summary["revenue_deviation_pct"].apply(
        lambda x: f"{x:+.1f}%"
    )

    return summary


# Test anomaly detection
print("\nRunning anomaly detection on monthly data...")
df_monthly  = load_anomaly_data("monthly")
df_detected = detect_anomalies(df_monthly)
anomaly_summary = summarize_anomalies(df_detected)

print(f"\n🚨 ANOMALIES DETECTED: {df_detected['is_anomaly'].sum()} months flagged")
print(f"   Out of {len(df_detected)} total months\n")
print("Anomaly Summary:")
print(anomaly_summary.to_string(index=False))



Running anomaly detection on monthly data...

🚨 ANOMALIES DETECTED: 4 months flagged
   Out of 62 total months

Anomaly Summary:
  period dimension    revenue  order_count revenue_deviation_pct   severity
Apr 2017   Overall    $30,832           26                -89.5% 🔴 Critical
Apr 2016   Overall    $80,858           40                -72.2% 🔴 Critical
Feb 2020   Overall $1,760,645         1107                 -5.3%     🟠 High
Dec 2019   Overall $2,045,390         1241                +30.2% 🔴 Critical


In [ ]:

# ─── CELL 17: ANOMALY VISUALISATION ──────────────────────────────────────────

def plot_anomalies(
    df:    pd.DataFrame,
    title: str = "Anomaly Detection — Monthly Revenue"
) -> go.Figure:
    """
    Plot revenue time series with anomalies highlighted in red.
    """

    df       = df.sort_values("period").copy()
    normal   = df[df["is_anomaly"] == False]
    anomalies= df[df["is_anomaly"] == True]

    # Convert dates to strings
    all_dates      = df["period"].dt.strftime("%Y-%m-%d").tolist()
    normal_dates   = normal["period"].dt.strftime("%Y-%m-%d").tolist()
    anomaly_dates  = anomalies["period"].dt.strftime("%Y-%m-%d").tolist()

    fig = make_subplots(
        rows = 2, cols = 1,
        subplot_titles = (
            title,
            "Anomaly Score (lower = more anomalous)"
        ),
        row_heights      = [0.65, 0.35],
        vertical_spacing = 0.12,
    )

    # ── Rolling mean line ─────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x    = all_dates,
        y    = df["rolling_mean"].tolist(),
        mode = "lines",
        name = "3-Month Rolling Avg",
        line = dict(color="#94A3B8", width=1.5, dash="dot"),
    ), row=1, col=1)

    # ── Normal revenue points ─────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x      = normal_dates,
        y      = normal["revenue"].tolist(),
        mode   = "lines+markers",
        name   = "Normal Revenue",
        line   = dict(color="#2563EB", width=2),
        marker = dict(size=6, color="#2563EB"),
    ), row=1, col=1)

    # ── Anomaly points ────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x    = anomaly_dates,
        y    = anomalies["revenue"].tolist(),
        mode = "markers+text",
        name = "⚠️ Anomaly",
        marker = dict(
            size   = 14,
            color  = "#EF4444",
            symbol = "x",
            line   = dict(width=2, color="#7F1D1D"),
        ),
        text          = anomalies["revenue_deviation_pct"].apply(
            lambda x: f"{x:+.0f}%"
        ).tolist(),
        textposition  = "top center",
        textfont      = dict(color="#EF4444", size=10),
    ), row=1, col=1)

    # ── Anomaly score bar chart ───────────────────────────────────────────────
    bar_colors = [
        "#EF4444" if is_anom else "#93C5FD"
        for is_anom in df["is_anomaly"].tolist()
    ]

    fig.add_trace(go.Bar(
        x      = all_dates,
        y      = df["anomaly_raw"].tolist(),
        name   = "Anomaly Score",
        marker = dict(color=bar_colors),
    ), row=2, col=1)

    # Threshold line on score chart
    fig.add_hline(
        y          = df[df["is_anomaly"] == True]["anomaly_raw"].max()
                     if df["is_anomaly"].any() else -0.5,
        line_dash  = "dash",
        line_color = "#EF4444",
        annotation_text = "Anomaly Threshold",
        row = 2, col = 1,
    )

    # ── Layout ────────────────────────────────────────────────────────────────
    n_anomalies = df["is_anomaly"].sum()

    fig.update_layout(
        title = dict(
            text = (
                f"🚨 Anomaly Detection — {title}<br>"
                f"<sup>{n_anomalies} anomalies detected "
                f"out of {len(df)} data points "
                f"({n_anomalies/len(df)*100:.1f}% contamination rate)</sup>"
            ),
            font = dict(size=15),
        ),
        height        = 650,
        hovermode     = "x unified",
        legend        = dict(orientation="h", yanchor="bottom", y=1.02),
        plot_bgcolor  = "white",
        paper_bgcolor = "white",
    )

    fig.update_yaxes(title_text="Revenue (USD)", tickformat="$,.0f", row=1, col=1)
    fig.update_yaxes(title_text="Isolation Score",                   row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)

    fig.show()
    return fig


# Plot monthly anomalies
fig = plot_anomalies(df_detected, "Monthly Revenue — Overall")
print("✅ Anomaly chart displayed")


✅ Anomaly chart displayed


In [ ]:

# ─── CELL 18: MULTI-DIMENSION ANOMALY SCANNER ────────────────────────────────

def full_anomaly_scan() -> dict:
    """
    Scan for anomalies across all dimensions:
    Overall, by Category, by Store Country
    Returns summary of all findings.
    """

    print("\n" + "="*60)
    print("🔍 FULL ANOMALY SCAN — All Dimensions")
    print("="*60)

    all_findings = {}

    # ── 1. Overall monthly ────────────────────────────────────────────────────
    print("\n📊 Scanning: Overall Monthly Revenue...")
    df_overall  = load_anomaly_data("monthly")
    df_overall  = detect_anomalies(df_overall)
    n           = df_overall["is_anomaly"].sum()
    print(f"   → {n} anomalies found")
    all_findings["overall"] = df_overall

    # ── 2. By category ────────────────────────────────────────────────────────
    print("\n📦 Scanning: Revenue by Category...")
    df_cat = load_anomaly_data("category")

    cat_results = {}
    for cat in df_cat["dimension"].unique():
        df_c = detect_anomalies(df_cat, dimension_filter=cat)
        n    = df_c["is_anomaly"].sum()
        if n > 0:
            cat_results[cat] = df_c
            print(f"   → {cat}: {n} anomalies")
    all_findings["category"] = cat_results

    # ── 3. By store country ───────────────────────────────────────────────────
    print("\n🏪 Scanning: Revenue by Store Country...")
    df_store = load_anomaly_data("store")

    # Aggregate to country level for cleaner signal
    df_country = df_store.groupby(
        [pd.Grouper(key="period", freq="MS"),
         df_store["dimension"].str.split(" - ").str[0]]
    ).agg({
        "revenue":          "sum",
        "order_count":      "sum",
        "total_units":      "sum",
        "avg_margin":       "mean",
        "unique_customers": "sum",
    }).reset_index()
    df_country.columns = ["period","dimension","revenue","order_count",
                          "total_units","avg_margin","unique_customers"]

    country_results = {}
    for country in df_country["dimension"].unique():
        df_c = detect_anomalies(df_country, dimension_filter=country)
        n    = df_c["is_anomaly"].sum()
        if n > 0:
            country_results[country] = df_c
            print(f"   → {country}: {n} anomalies")
    all_findings["country"] = country_results

    # ── Final summary ─────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("📋 ANOMALY SCAN COMPLETE — SUMMARY")
    print("="*60)

    total = (
        all_findings["overall"]["is_anomaly"].sum() +
        sum(v["is_anomaly"].sum() for v in cat_results.values()) +
        sum(v["is_anomaly"].sum() for v in country_results.values())
    )

    print(f"\nTotal anomalies detected: {total}")
    print(f"  Overall level  : {all_findings['overall']['is_anomaly'].sum()}")
    print(f"  Category level : {sum(v['is_anomaly'].sum() for v in cat_results.values())}")
    print(f"  Country level  : {sum(v['is_anomaly'].sum() for v in country_results.values())}")

    # ── Top critical findings ─────────────────────────────────────────────────
    print("\n🔴 TOP CRITICAL FINDINGS:")

    all_anomaly_rows = []

    # Overall
    ovr_anom = all_findings["overall"][
        all_findings["overall"]["is_anomaly"] == True
    ].copy()
    ovr_anom["dimension"] = "Overall"
    all_anomaly_rows.append(ovr_anom)

    # Category
    for cat, df_c in cat_results.items():
        anom = df_c[df_c["is_anomaly"] == True].copy()
        anom["dimension"] = f"Category: {cat}"
        all_anomaly_rows.append(anom)

    # Country
    for country, df_c in country_results.items():
        anom = df_c[df_c["is_anomaly"] == True].copy()
        anom["dimension"] = f"Country: {country}"
        all_anomaly_rows.append(anom)

    if all_anomaly_rows:
        combined = pd.concat(all_anomaly_rows, ignore_index=True)
        combined = combined.sort_values("revenue_deviation_pct").head(10)
        combined["period"]   = combined["period"].dt.strftime("%b %Y")
        combined["revenue"]  = combined["revenue"].apply(lambda x: f"${x:,.0f}")
        combined["revenue_deviation_pct"] = combined["revenue_deviation_pct"].apply(
            lambda x: f"{x:+.1f}%"
        )
        print(combined[[
            "period","dimension","revenue",
            "revenue_deviation_pct","severity"
        ]].to_string(index=False))

    return all_findings


# Run full scan
findings = full_anomaly_scan()



🔍 FULL ANOMALY SCAN — All Dimensions

📊 Scanning: Overall Monthly Revenue...
   → 4 anomalies found

📦 Scanning: Revenue by Category...
   → Audio: 4 anomalies
   → Cameras and camcorders: 4 anomalies
   → Cell phones: 4 anomalies
   → Computers: 4 anomalies
   → Games and Toys: 4 anomalies
   → Home Appliances: 4 anomalies
   → Music, Movies and Audio Books: 4 anomalies
   → TV and Video: 4 anomalies

🏪 Scanning: Revenue by Store Country...
   → Australia: 4 anomalies
   → Canada: 4 anomalies
   → France: 3 anomalies
   → Germany: 4 anomalies
   → Italy: 3 anomalies
   → Netherlands: 3 anomalies
   → United Kingdom: 4 anomalies
   → United States: 4 anomalies

📋 ANOMALY SCAN COMPLETE — SUMMARY

Total anomalies detected: 65
  Overall level  : 4
  Category level : 32
  Country level  : 29

🔴 TOP CRITICAL FINDINGS:
  period                        dimension revenue revenue_deviation_pct   severity
Apr 2017          Country: United Kingdom    $516                -98.9% 🔴 Critical
Apr 2018

In [ ]:

# ─── CELL 19: INTERACTIVE ANOMALY EXPLORER ───────────────────────────────────

print("\n" + "="*60)
print("🔍 ANOMALY EXPLORER — Interactive Mode")
print("="*60)

print("\nSelect analysis level:")
print("  1 → Overall monthly revenue")
print("  2 → By product category")
print("  3 → By store country")

choice = input("\nSelect (1-3): ").strip()

if choice == "1":
    df_plot = findings["overall"]
    title   = "Monthly Revenue — Overall"
    fig     = plot_anomalies(df_plot, title)

elif choice == "2":
    categories = list(findings["category"].keys())
    if not categories:
        print("No category anomalies found.")
    else:
        print("\nCategories with anomalies:")
        for i, cat in enumerate(categories, 1):
            n = findings["category"][cat]["is_anomaly"].sum()
            print(f"  {i} → {cat} ({n} anomalies)")

        cat_choice = input("\nSelect category (number): ").strip()
        try:
            selected_cat = categories[int(cat_choice) - 1]
            df_plot      = findings["category"][selected_cat]
            title        = f"Category: {selected_cat}"
            fig          = plot_anomalies(df_plot, title)
        except Exception as e:
            print(f"Invalid selection: {e}")

elif choice == "3":
    countries = list(findings["country"].keys())
    if not countries:
        print("No country anomalies found.")
    else:
        print("\nCountries with anomalies:")
        for i, country in enumerate(countries, 1):
            n = findings["country"][country]["is_anomaly"].sum()
            print(f"  {i} → {country} ({n} anomalies)")

        country_choice = input("\nSelect country (number): ").strip()
        try:
            selected_country = countries[int(country_choice) - 1]
            df_plot          = findings["country"][selected_country]
            title            = f"Country: {selected_country}"
            fig              = plot_anomalies(df_plot, title)
        except Exception as e:
            print(f"Invalid selection: {e}")



🔍 ANOMALY EXPLORER — Interactive Mode

Select analysis level:
  1 → Overall monthly revenue
  2 → By product category
  3 → By store country

Select (1-3): 3

Countries with anomalies:
  1 → Australia (4 anomalies)
  2 → Canada (4 anomalies)
  3 → France (3 anomalies)
  4 → Germany (4 anomalies)
  5 → Italy (3 anomalies)
  6 → Netherlands (3 anomalies)
  7 → United Kingdom (4 anomalies)
  8 → United States (4 anomalies)

Select country (number): 3


In [ ]:

# ─── CELL 20: AI-POWERED ANOMALY EXPLANATION ─────────────────────────────────
# Uses Groq LLM to explain WHY anomalies happened in business terms

def explain_anomalies_with_ai(df: pd.DataFrame, context: str = "") -> str:
    """
    Send anomaly findings to Groq LLM for business explanation.
    """

    anomalies = df[df["is_anomaly"] == True].copy()

    if anomalies.empty:
        return "No anomalies detected in this dataset."

    # Build anomaly summary for LLM
    anomaly_text = []
    for _, row in anomalies.iterrows():
        anomaly_text.append(
            f"- {row['period'].strftime('%b %Y')}: "
            f"Revenue ${row['revenue']:,.0f} "
            f"({row['revenue_deviation_pct']:+.1f}% vs rolling avg), "
            f"Orders: {int(row['order_count'])}, "
            f"Units: {int(row['total_units'])}"
        )

    prompt = f"""
You are a senior retail business analyst.

You detected the following revenue anomalies in a global electronics retail dataset:
{chr(10).join(anomaly_text)}

Context about this business:
- Global electronics retailer (Computers, Audio, TV, Phones, Cameras)
- Operates in US, UK, Canada, Australia, France, Germany, Netherlands, Italy
- Sales period: 2016-2020
- COVID-19 pandemic started March 2020
{context}

For each anomaly:
1. Suggest the most likely business reason
2. Rate it as SPIKE or DROP
3. Suggest one action the business should take

Keep response concise and business-focused.
"""

    response = groq_client.chat.completions.create(
        model       = "llama-3.3-70b-versatile",
        messages    = [{"role": "user", "content": prompt}],
        temperature = 0.4,
        max_tokens  = 600,
    )

    return response.choices[0].message.content.strip()


print("\n" + "="*60)
print("🤖 AI ANOMALY EXPLANATION")
print("="*60)
print("🔄 Asking AI to explain the anomalies...")

explanation = explain_anomalies_with_ai(
    df_detected,
    context="This is overall revenue across all categories and countries."
)

print(f"\n💡 AI Analysis:\n{explanation}")


🤖 AI ANOMALY EXPLANATION
🔄 Asking AI to explain the anomalies...

💡 AI Analysis:
Here are the analyses for each anomaly:

1. Apr 2016: 
   - Likely reason: Easter holiday sales slump or inventory management issue
   - Rating: DROP
   - Action: Review inventory management and sales strategies for holiday periods

2. Apr 2017: 
   - Likely reason: Similar issue as Apr 2016, possibly exacerbated by market trends
   - Rating: DROP
   - Action: Analyze market trends and competitor sales during the same period

3. Dec 2019: 
   - Likely reason: Holiday season sales boost (e.g., Christmas)
   - Rating: SPIKE
   - Action: Scale up marketing and inventory for future holiday seasons

4. Feb 2020: 
   - Likely reason: Initial COVID-19 pandemic impact on sales and supply chain
   - Rating: DROP
   - Action: Develop a pandemic response plan, including supply chain risk management and online sales strategies


# AUTO INSIGHT GENERATOR

In [ ]:
# =============================================================================
# RETAIL AI INTELLIGENCE PLATFORM — MODULE 4: AUTO INSIGHT GENERATOR
# Paste these as NEW cells AFTER Modules 1, 2, and 3 in Colab
# Requires: Module 1 cells already run (Snowflake connection + Groq client)
# =============================================================================

# ─── CELL 21: DATA COLLECTOR FOR INSIGHTS ────────────────────────────────────

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

def collect_business_snapshot() -> dict:
    """
    Pull all key metrics from Snowflake mart tables
    to build a complete business snapshot for the AI.
    Returns a dictionary of DataFrames and metrics.
    """

    print("🔄 Collecting business snapshot from Snowflake...")
    snapshot = {}

    # ── 1. Overall KPIs ───────────────────────────────────────────────────────
    kpi_sql = """
        SELECT
            COUNT(DISTINCT order_number)             AS total_orders,
            ROUND(SUM(revenue_usd), 2)               AS total_revenue,
            ROUND(SUM(profit_usd), 2)                AS total_profit,
            ROUND(AVG(margin_pct), 2)                AS avg_margin_pct,
            SUM(quantity)                            AS total_units,
            COUNT(DISTINCT customer_key)             AS unique_customers,
            ROUND(SUM(revenue_usd) /
                  COUNT(DISTINCT order_number), 2)   AS avg_order_value,
            MIN(order_date)                          AS first_order_date,
            MAX(order_date)                          AS last_order_date
        FROM RAW_MARTS.MART_SALES_FACT
    """
    snapshot["kpis"] = run_query(kpi_sql).iloc[0].to_dict()
    print("  ✅ KPIs collected")

    # ── 2. Revenue by year ────────────────────────────────────────────────────
    yearly_sql = """
        SELECT
            order_year                               AS year,
            ROUND(SUM(revenue_usd), 2)               AS revenue,
            COUNT(DISTINCT order_number)             AS orders,
            COUNT(DISTINCT customer_key)             AS customers
        FROM RAW_MARTS.MART_SALES_FACT
        GROUP BY order_year
        ORDER BY order_year
    """
    snapshot["yearly"] = run_query(yearly_sql)
    print("  ✅ Yearly revenue collected")

    # ── 3. Revenue by category ────────────────────────────────────────────────
    category_sql = """
        SELECT
            category,
            ROUND(SUM(revenue_usd), 2)               AS revenue,
            ROUND(SUM(profit_usd), 2)                AS profit,
            ROUND(AVG(margin_pct), 2)                AS avg_margin,
            COUNT(DISTINCT order_number)             AS orders,
            SUM(quantity)                            AS units
        FROM RAW_MARTS.MART_SALES_FACT
        GROUP BY category
        ORDER BY revenue DESC
    """
    snapshot["categories"] = run_query(category_sql)
    print("  ✅ Category data collected")

    # ── 4. Revenue by country ─────────────────────────────────────────────────
    country_sql = """
        SELECT
            store_country                            AS country,
            ROUND(SUM(revenue_usd), 2)               AS revenue,
            COUNT(DISTINCT order_number)             AS orders,
            COUNT(DISTINCT customer_key)             AS customers,
            ROUND(AVG(margin_pct), 2)                AS avg_margin
        FROM RAW_MARTS.MART_SALES_FACT
        GROUP BY store_country
        ORDER BY revenue DESC
    """
    snapshot["countries"] = run_query(country_sql)
    print("  ✅ Country data collected")

    # ── 5. Channel split ──────────────────────────────────────────────────────
    channel_sql = """
        SELECT
            channel,
            ROUND(SUM(revenue_usd), 2)               AS revenue,
            COUNT(DISTINCT order_number)             AS orders,
            ROUND(SUM(revenue_usd) /
                  COUNT(DISTINCT order_number), 2)   AS avg_order_value
        FROM RAW_MARTS.MART_SALES_FACT
        GROUP BY channel
        ORDER BY revenue DESC
    """
    snapshot["channels"] = run_query(channel_sql)
    print("  ✅ Channel data collected")

    # ── 6. RFM segment summary ────────────────────────────────────────────────
    rfm_sql = """
        SELECT
            rfm_segment,
            COUNT(*)                                 AS customer_count,
            ROUND(AVG(total_revenue_usd), 2)         AS avg_clv,
            ROUND(AVG(total_orders), 2)              AS avg_orders,
            ROUND(AVG(days_since_last_order), 0)     AS avg_recency_days
        FROM RAW_MARTS.MART_DIM_CUSTOMER
        WHERE has_orders = TRUE
          AND rfm_segment IS NOT NULL
        GROUP BY rfm_segment
        ORDER BY avg_clv DESC
    """
    snapshot["rfm"] = run_query(rfm_sql)
    print("  ✅ RFM segments collected")

    # ── 7. Top 5 products ─────────────────────────────────────────────────────
    products_sql = """
        SELECT
            product_name,
            category,
            ROUND(total_revenue_usd, 2)              AS revenue,
            total_units_sold                         AS units,
            ROUND(list_margin_pct, 2)                AS margin_pct,
            abc_class
        FROM RAW_MARTS.MART_DIM_PRODUCT
        ORDER BY total_revenue_usd DESC
        LIMIT 5
    """
    snapshot["top_products"] = run_query(products_sql)
    print("  ✅ Top products collected")

    # ── 8. Store performance ──────────────────────────────────────────────────
    store_sql = """
        SELECT
            country,
            store_type,
            COUNT(*)                                 AS store_count,
            ROUND(SUM(total_revenue_usd), 2)         AS total_revenue,
            ROUND(AVG(revenue_per_sqm), 2)           AS avg_rev_per_sqm
        FROM RAW_MARTS.MART_DIM_STORE
        GROUP BY country, store_type
        ORDER BY total_revenue DESC
        LIMIT 8
    """
    snapshot["stores"] = run_query(store_sql)
    print("  ✅ Store data collected")

    # ── 9. YoY growth from executive summary ─────────────────────────────────
    yoy_sql = """
        SELECT
            year,
            ROUND(SUM(revenue_usd), 2)               AS revenue,
            ROUND(AVG(yoy_growth_pct), 2)            AS avg_yoy_growth
        FROM RAW_MARTS.MART_EXECUTIVE_SUMMARY
        WHERE yoy_growth_pct IS NOT NULL
        GROUP BY year
        ORDER BY year
    """
    snapshot["yoy"] = run_query(yoy_sql)
    print("  ✅ YoY growth collected")

    print("\n✅ Business snapshot complete — all data collected!")
    return snapshot


# Collect snapshot
snapshot = collect_business_snapshot()



🔄 Collecting business snapshot from Snowflake...
  ✅ KPIs collected
  ✅ Yearly revenue collected
  ✅ Category data collected
  ✅ Country data collected
  ✅ Channel data collected
  ✅ RFM segments collected
  ✅ Top products collected
  ✅ Store data collected
  ✅ YoY growth collected

✅ Business snapshot complete — all data collected!


In [ ]:

# ─── CELL 22: INSIGHT PROMPT BUILDER ─────────────────────────────────────────

def build_insight_prompt(
    snapshot:      dict,
    insight_type:  str = "executive",
    focus_area:    str = None,
    time_period:   str = "full"
) -> str:
    """
    Build a structured prompt for the AI insight generator.

    Args:
        snapshot:     business data snapshot from Cell 21
        insight_type: 'executive' | 'category' | 'customer' | 'store' | 'weekly'
        focus_area:   specific category/country to focus on
        time_period:  'full' | '2019' | '2020'
    """

    kpis       = snapshot["kpis"]
    yearly_df  = snapshot["yearly"]
    cat_df     = snapshot["categories"]
    country_df = snapshot["countries"]
    channel_df = snapshot["channels"]
    rfm_df     = snapshot["rfm"]
    prod_df    = snapshot["top_products"]
    store_df   = snapshot["stores"]
    yoy_df     = snapshot["yoy"]

    # ── Format data sections ──────────────────────────────────────────────────
    yearly_text = yearly_df.to_string(index=False)
    cat_text    = cat_df.head(5).to_string(index=False)
    country_text= country_df.head(5).to_string(index=False)
    channel_text= channel_df.to_string(index=False)
    rfm_text    = rfm_df.to_string(index=False)
    prod_text   = prod_df.to_string(index=False)
    store_text  = store_df.head(5).to_string(index=False)
    yoy_text    = yoy_df.to_string(index=False)

    # ── Base context ──────────────────────────────────────────────────────────
    base_context = f"""
You are a senior retail business analyst writing for C-suite executives.
You have access to complete sales data for a global electronics retailer.

BUSINESS OVERVIEW:
- Total Revenue:     ${float(kpis.get('total_revenue', 0)):,.0f}
- Total Profit:      ${float(kpis.get('total_profit', 0)):,.0f}
- Avg Gross Margin:  {float(kpis.get('avg_margin_pct', 0)):.1f}%
- Total Orders:      {int(float(kpis.get('total_orders', 0))):,}
- Avg Order Value:   ${float(kpis.get('avg_order_value', 0)):,.0f}
- Unique Customers:  {int(float(kpis.get('unique_customers', 0))):,}
- Data Period:       2016 - 2021

ANNUAL REVENUE:
{yearly_text}

YEAR-OVER-YEAR GROWTH:
{yoy_text}

TOP CATEGORIES BY REVENUE:
{cat_text}

TOP COUNTRIES BY REVENUE:
{country_text}

CHANNEL PERFORMANCE:
{channel_text}

TOP 5 PRODUCTS:
{prod_text}

RFM CUSTOMER SEGMENTS:
{rfm_text}

STORE PERFORMANCE:
{store_text}
"""

    # ── Prompt by insight type ────────────────────────────────────────────────
    if insight_type == "executive":
        prompt = base_context + """

TASK: Write a comprehensive Executive Business Intelligence Report.

Structure it exactly like this:

## 📊 EXECUTIVE SUMMARY
[3-4 sentences: overall business health, key achievement, main concern]

## 💰 REVENUE PERFORMANCE
[Analyze revenue trends year by year. Explain the 2018 surge, 2019 peak, 2020 decline. Use actual numbers.]

## 🏆 TOP PERFORMERS
[Which categories, countries, products are driving growth. Be specific with numbers.]

## ⚠️ RISKS & CONCERNS
[Revenue concentration risks, declining trends, underperforming areas]

## 👥 CUSTOMER INSIGHTS
[RFM segment analysis — who are our best customers, who is at risk]

## 💡 STRATEGIC RECOMMENDATIONS
[5 specific, actionable recommendations with expected business impact]

## 🔮 OUTLOOK
[Forward looking statement based on trends]

Use actual numbers throughout. Be specific, concise, and business-focused.
Write in a professional but direct tone suitable for a board presentation.
"""

    elif insight_type == "category":
        prompt = base_context + f"""

TASK: Write a deep-dive Category Performance Report{f' focused on {focus_area}' if focus_area else ''}.

Structure:
## CATEGORY OVERVIEW
## TOP PERFORMING CATEGORIES
## UNDERPERFORMING CATEGORIES
## MARGIN ANALYSIS
## RECOMMENDATIONS FOR CATEGORY MIX

Use actual numbers. Focus on actionable insights.
"""

    elif insight_type == "customer":
        prompt = base_context + """

TASK: Write a Customer Analytics Report.

Structure:
## CUSTOMER BASE OVERVIEW
## RFM SEGMENT ANALYSIS (explain each segment with numbers)
## HIGH VALUE CUSTOMERS (Champions + Loyal)
## AT RISK CUSTOMERS (At Risk + Hibernating + Lost)
## REACTIVATION OPPORTUNITIES
## CUSTOMER RETENTION RECOMMENDATIONS

Use actual numbers. Focus on actionable insights.
"""

    elif insight_type == "store":
        prompt = base_context + """

TASK: Write a Store Performance Report.

Structure:
## STORE NETWORK OVERVIEW
## TOP PERFORMING MARKETS
## ONLINE VS PHYSICAL ANALYSIS
## STORE EFFICIENCY (revenue per sqm)
## EXPANSION RECOMMENDATIONS

Use actual numbers. Focus on actionable insights.
"""

    elif insight_type == "weekly":
        prompt = base_context + f"""

TASK: Write a concise Weekly Business Briefing (max 300 words).

Format:
🟢 WINS THIS PERIOD:
[Top 3 positive highlights with numbers]

🔴 CONCERNS:
[Top 2 areas needing attention]

📋 KEY METRICS:
[5 bullet KPIs]

🎯 FOCUS FOR NEXT PERIOD:
[2 specific actions]

Keep it brief, direct, and numbers-driven.
"""

    return prompt


print("✅ Insight prompt builder ready")



✅ Insight prompt builder ready


In [ ]:

# ─── CELL 23: AI INSIGHT GENERATOR ───────────────────────────────────────────

def generate_insight_report(
    snapshot:     dict,
    insight_type: str   = "executive",
    focus_area:   str   = None,
    max_tokens:   int   = 1500
) -> str:
    """
    Generate an AI-written business insight report.

    Args:
        snapshot:     data from collect_business_snapshot()
        insight_type: 'executive' | 'category' | 'customer' | 'store' | 'weekly'
        focus_area:   optional specific area to focus on
        max_tokens:   length of report (1500 = ~1 page)

    Returns:
        Formatted report as a string
    """

    report_names = {
        "executive": "Executive Business Intelligence Report",
        "category":  "Category Performance Report",
        "customer":  "Customer Analytics Report",
        "store":     "Store Performance Report",
        "weekly":    "Weekly Business Briefing",
    }

    report_name = report_names.get(insight_type, "Business Report")

    print(f"\n{'='*60}")
    print(f"📄 GENERATING: {report_name}")
    if focus_area:
        print(f"   Focus: {focus_area}")
    print(f"{'='*60}")
    print("🔄 AI is writing your report...\n")

    # Build prompt
    prompt = build_insight_prompt(snapshot, insight_type, focus_area)

    # Call Groq
    response = groq_client.chat.completions.create(
        model       = "llama-3.3-70b-versatile",
        messages    = [{"role": "user", "content": prompt}],
        temperature = 0.4,
        max_tokens  = max_tokens,
    )

    report = response.choices[0].message.content.strip()

    # Print formatted report
    print("=" * 60)
    print(f"  {report_name.upper()}")
    print(f"  Generated: {datetime.now().strftime('%d %b %Y %H:%M')}")
    print("=" * 60)
    print(report)
    print("\n" + "=" * 60)

    return report


# Generate Executive Report first
executive_report = generate_insight_report(
    snapshot     = snapshot,
    insight_type = "executive",
    max_tokens   = 1500
)




📄 GENERATING: Executive Business Intelligence Report
🔄 AI is writing your report...

  EXECUTIVE BUSINESS INTELLIGENCE REPORT
  Generated: 06 Jun 2026 22:08
## 📊 EXECUTIVE SUMMARY
Our global electronics retailer has achieved a total revenue of $44,611,750 with a total profit of $25,312,592, representing an average gross margin of 54.5%. The key achievement is the significant revenue growth from 2016 to 2019, with a peak of $14,975,595 in 2019. However, the main concern is the substantial decline in revenue in 2020 and 2021, with a 67.5% year-over-year decrease in 2021. This decline necessitates a thorough analysis and strategic planning to revitalize business growth.

## 💰 REVENUE PERFORMANCE
Analyzing revenue trends year by year, we observe a steady increase from $4,996,555 in 2016 to $10,344,238 in 2018, representing a 107% surge. This surge can be attributed to the expansion of product categories and entry into new markets. The revenue peaked at $14,975,595 in 2019, with a year-ove

In [ ]:

# ─── CELL 24: ALL REPORT TYPES GENERATOR ─────────────────────────────────────

print("\n" + "="*60)
print("📄 AUTO INSIGHT GENERATOR — Interactive Mode")
print("="*60)

print("\nSelect report type:")
print("  1 → Executive Report      (full business overview)")
print("  2 → Category Report       (product performance deep dive)")
print("  3 → Customer Report       (RFM + CLV analysis)")
print("  4 → Store Report          (geographic performance)")
print("  5 → Weekly Briefing       (short summary, key numbers)")
print("  6 → Generate ALL reports  (all 5 at once)")

choice = input("\nSelect report (1-6): ").strip()

report_map = {
    "1": "executive",
    "2": "category",
    "3": "customer",
    "4": "store",
    "5": "weekly",
}

if choice == "6":
    # Generate all reports
    all_reports = {}
    for key, rtype in report_map.items():
        print(f"\n{'='*60}")
        print(f"Generating {rtype.upper()} report...")
        all_reports[rtype] = generate_insight_report(
            snapshot     = snapshot,
            insight_type = rtype,
            max_tokens   = 800 if rtype == "weekly" else 1200,
        )
    print("\n✅ All 5 reports generated!")

elif choice in report_map:
    report_type = report_map[choice]
    report = generate_insight_report(
        snapshot     = snapshot,
        insight_type = report_type,
        max_tokens   = 800 if report_type == "weekly" else 1500,
    )

else:
    print("Invalid choice — generating executive report by default")
    report = generate_insight_report(snapshot, "executive")



📄 AUTO INSIGHT GENERATOR — Interactive Mode

Select report type:
  1 → Executive Report      (full business overview)
  2 → Category Report       (product performance deep dive)
  3 → Customer Report       (RFM + CLV analysis)
  4 → Store Report          (geographic performance)
  5 → Weekly Briefing       (short summary, key numbers)
  6 → Generate ALL reports  (all 5 at once)

Select report (1-6): 4

📄 GENERATING: Store Performance Report
🔄 AI is writing your report...

  STORE PERFORMANCE REPORT
  Generated: 06 Jun 2026 22:08
## STORE NETWORK OVERVIEW
The global electronics retailer operates a network of 46 stores across 4 countries, with 24 physical stores in the United States, 7 in the United Kingdom, 9 in Germany, and 5 in Canada. In addition to the physical stores, the company also has an online store. The total revenue generated by the store network is $44,611,750, with an average revenue per square meter of $115.29 for physical stores.

## TOP PERFORMING MARKETS
The top perfo

In [ ]:


# ─── CELL 25: SAVE REPORTS TO FILE ───────────────────────────────────────────

def save_report_to_file(report: str, report_type: str) -> str:
    """Save generated report to a text file in Colab."""

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    filename  = f"retail_insight_{report_type}_{timestamp}.txt"

    header = f"""
================================================================================
RETAIL AI INTELLIGENCE PLATFORM
{report_type.upper()} REPORT
Generated: {datetime.now().strftime('%d %B %Y at %H:%M')}
Data Source: Snowflake — RETAIL_RAW.RAW_MARTS
================================================================================

"""

    with open(filename, "w") as f:
        f.write(header + report)

    print(f"✅ Report saved to: {filename}")
    print(f"   Download from Colab Files panel (left sidebar → Files icon)")
    return filename


# Save the executive report
saved_file = save_report_to_file(executive_report, "executive")


✅ Report saved to: retail_insight_executive_20260606_2208.txt
   Download from Colab Files panel (left sidebar → Files icon)


In [ ]:


# ─── CELL 26: SCHEDULED INSIGHT SUMMARY ──────────────────────────────────────
# Simulates what a daily/weekly automated run would produce

def generate_daily_briefing(snapshot: dict) -> str:
    """
    Generate a concise daily briefing —
    designed to be emailed or sent to Slack automatically.
    """

    print("\n" + "="*60)
    print("📬 DAILY BRIEFING GENERATOR")
    print("="*60)

    briefing = generate_insight_report(
        snapshot     = snapshot,
        insight_type = "weekly",
        max_tokens   = 600,
    )

    print("\n📧 This briefing could be:")
    print("   → Emailed daily via Python SMTP")
    print("   → Posted to Slack via webhook")
    print("   → Saved to SharePoint automatically")
    print("   → Triggered by Airflow/cron job")

    return briefing


daily_briefing = generate_daily_briefing(snapshot)




📬 DAILY BRIEFING GENERATOR

📄 GENERATING: Weekly Business Briefing
🔄 AI is writing your report...

  WEEKLY BUSINESS BRIEFING
  Generated: 06 Jun 2026 22:09
🟢 WINS THIS PERIOD:
1. Total Revenue: $44,611,750, with an average gross margin of 54.5%.
2. Top-performing category: Computers, generating $16,623,225 in revenue with a 54.05% margin.
3. Strong online presence: $9,137,412 in revenue from online channels, with an average order value of $1,658.

🔴 CONCERNS:
1. Year-over-year growth decline: -67.50% in 2021, indicating a need for strategic adjustment.
2. Customer segment analysis: 1,878 "Lost" customers, with an average CLV of $8,312, highlighting the need for targeted retention efforts.

📋 KEY METRICS:
* Total Orders: 26,006
* Average Order Value: $1,715
* Unique Customers: 11,839
* Average Gross Margin: 54.5%
* Total Profit: $25,312,592

🎯 FOCUS FOR NEXT PERIOD:
1. Develop targeted marketing campaigns to re-engage "Lost" customers and improve year-over-year growth.
2. Analyze and 

# NOTES

Module 1 — AI Chat Assistant

The chat assistant works by combining two things — a Large Language Model and your Snowflake database. When you type a question like "Which store had the highest revenue?", the question is first sent to the Groq API which runs the Llama 3.3 70B model. The model has been given a detailed description of all your mart tables, their columns, and their relationships as context. Using this context it generates a valid Snowflake SQL query that answers your question. That SQL is then executed directly against your MART_SALES_FACT, MART_DIM_STORE, MART_DIM_PRODUCT and other mart tables using the snowflake-connector-python library. The results come back as a pandas DataFrame. That DataFrame is then sent back to the LLM a second time, and the model writes a human-readable business answer using the actual numbers. So every answer you get is grounded in real data — the AI never makes up numbers.

Module 2 — Sales Forecasting Engine

The forecasting module pulls monthly aggregated revenue from your MART_SALES_FACT table grouped by month using a Snowflake SQL query. This gives a time series of revenue from January 2016 to December 2020. That time series is fed into Meta's Prophet model — an open source forecasting library specifically designed for business time series with seasonality and trend changes. Prophet fits a mathematical model to your historical data learning three things — the overall growth trend, yearly seasonality patterns like holiday spikes, and the COVID-19 lockdown period which we explicitly told it about as a known shock event. Once trained, Prophet predicts the next 6 months of revenue along with 95% confidence intervals showing the best and worst case range. The results are visualized using Plotly which creates the interactive chart with the orange forecast line, confidence band, and seasonality subplot. The MAPE metric tells you how accurate the model was on historical data — lower is better.

Module 3 — Anomaly Detection Engine

The anomaly detection module uses a machine learning algorithm called Isolation Forest from the scikit-learn library. The idea behind Isolation Forest is simple — normal data points are hard to isolate because they cluster together, while anomalies are easy to isolate because they are different from everything else. The module first pulls monthly revenue data from MART_SALES_FACT aggregated at different levels — overall, by category, by country. For each data point it uses four features: revenue, order count, total units, and average margin. These four features are scaled using StandardScaler so no single feature dominates. The Isolation Forest then assigns each month an anomaly score — the lower the score the more anomalous that month is. Any month scoring below the threshold gets flagged as an anomaly with a severity label of Critical, High, Medium, or Normal. The revenue deviation percentage tells you how far that month's revenue was from its 3-month rolling average. Finally the detected anomalies are sent to Groq's Llama model which explains each anomaly in business terms — whether it was a COVID impact, a holiday spike, or a seasonal pattern.

Module 4 — Auto Insight Generator

The insight generator works like an automated analyst that reads all your data and writes a report. It starts by running nine separate SQL queries against your Snowflake mart tables — pulling overall KPIs from MART_SALES_FACT, yearly revenue trends, category performance, country breakdown, channel split, RFM segments from MART_DIM_CUSTOMER, top products from MART_DIM_PRODUCT, store metrics from MART_DIM_STORE, and YoY growth from MART_EXECUTIVE_SUMMARY. All these results are collected into a Python dictionary called a snapshot. A structured prompt is then built that embeds all this real data into a detailed business context and gives the Llama 3.3 70B model specific instructions on what type of report to write — Executive Report, Category Report, Customer Report, Store Report, or Weekly Briefing. The model writes the full report using actual numbers from your data. The report is then saved to a downloadable text file. The daily briefing version of this is designed to be run automatically on a schedule — it could be triggered by a cron job or Apache Airflow and sent via email or Slack every morning.